# AuroraStay Hotels: Policy Question Answering with RAG

## Problem Statement

### Business Context

AuroraStay Hotels is a mid-sized hotel group operating 18 business and leisure properties across major cities in the USA, adding 3 to 4 new properties every year.

Hotel staff handle around 1,200 guest queries per week about room types, check-in/check-out, cancellation policies, refunds, breakfast, parking, pets, amenities, and additional charges.

Simple queries can be answered using existing FAQs. However, around 40% of queries are complex and require information from multiple documents, such as booking terms, cancellation policies, room information, and guest guidelines.

Manually combining this information takes 8 to 12 minutes per query and results in a 15% to 20% error rate because answers may be incomplete or incorrect.

As the hotel group grows, the number of properties and property-specific policies also increases. With 45% annual staff turnover, institutional knowledge is constantly lost.

### Key Risks

- **Faithfulness:** Avoid unsupported information.
- **Retrieval Completeness:** Retrieve all relevant information required to answer the query.
- **Retrieval Precision:** Retrieve relevant source chunks while minimizing irrelevant information.
- **Answer Relevancy:** Directly answer the guest's question.

A RAG system can retrieve relevant information from these documents and help staff provide accurate, source-grounded answers.

### Objective

Build a POC for a user interface where hotel staff can enter natural-language questions and receive answers based on the hotel's documents. The application should:

- Provide a simple UI where staff can enter their questions.
- Retrieve the most relevant information from the hotel's document collection.
- Generate an answer grounded only in the retrieved information.
- Display the supporting source information along with the answer.
- Reduce the number of queries that require manual lookup or escalation, rather than focusing on reducing the time spent handling each query.

**Note:** The objective at this stage is not to reduce query-handling time. The primary objective is to determine whether the system can reliably answer a meaningful proportion of queries without requiring manual effort.

### Success Criteria

| Business Objective | Technical Metric | Threshold |
|---|---|---|
| Avoid unsupported information | Faithfulness | >= 0.85 |
| Retrieve all relevant information | Contextual Recall | >= 0.80 |
| Retrieve relevant chunks | Contextual Precision | >= 0.75 |
| Answer the question asked | Answer Relevancy | >= 0.75 |
| Reduce the number of queries requiring manual searches for policy information | Queries Confidently Answered | >= 50% |

**Composite Score** = 0.35 × Faithfulness + 0.25 × Contextual Recall + 0.20 × Contextual Precision + 0.20 × Answer Relevancy

Faithfulness carries the highest weight because avoiding unsupported or misleading information is the most critical requirement in this use case.

Every generated answer is routed on its composite score:

- **Auto Answer (score >= 75%)**: the answer and its source chunks are reliable enough to present directly to the representative.
- **Manual Lookup (score < 75%)**: the representative falls back to the existing manual process or escalates to the back-office helpdesk.

### Data Description

| Document | Pages | Description |
|---|---|---|
| aurorastay_booking_terms_and_cancellation_policy.pdf | 5 | Rate plan rules (Flexible, Advance Purchase, member/promotional, group), cancellation deadlines, no-show charges, modifications, taxes and deposits, and a six-property quick-reference matrix |
| aurorastay_room_and_property_information.pdf | 5 | Room types and occupancy limits, check-in/check-out times, early arrival and late departure, amenities, parking and pet policies, service animals, accessibility requests |
| aurorastay_refunds_and_additional_charges_guide.pdf | 4 | Refund eligibility by scenario, itemized additional charges, incidentals authorization and release, and the five-step refund dispute workflow |
| aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf | 5 | Guest conduct rules, ADA accessibility and service animal handling, safety and privacy, the five-stage complaint resolution process, and the escalation matrix |

| File | Queries | Description |
|---|---|---|
| evaluation.csv | 10 | Benchmark queries with expected answers and relevant policy contexts |
| test.xlsx / test.csv | 10 | Held-out test queries with expected answers and relevant contexts |

## Approach

The notebook follows the rubric sections in order:

1. **Data Understanding**: convert the four policy PDFs to structured markdown with Docling and inspect their structure and tables.
2. **RAG System**: token-aware chunking sized to the embedding model, a FAISS vector store, a retrieval function, and a generation step with explicit grounding rules.
3. **Evaluation**: score the pipeline with four LLM-as-a-Judge metrics on the evaluation set, then on the held-out test set, and compare against the success criteria.
4. **Deployment**: a Flask backend and Streamlit frontend, each with a Dockerfile, pushed to GitHub and run in Codespaces.
5. **Inference**: call the deployed API and inspect the answer and its supporting chunks.
6. **Business Insights and Recommendations**.

### Files needed before running

Place these next to the notebook (in Colab, upload them with the Files pane):

```
config.json                      <- {"OPENAI_API_KEY": "...", "OPENAI_API_BASE": "..."}
Data/
    aurorastay_room_and_property_information.pdf
    aurorastay_refunds_and_additional_charges_guide.pdf
    aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf
    aurorastay_booking_terms_and_cancellation_policy.pdf
    evaluation.csv
    test.csv   (or test.xlsx)
```

The notebook runs in both Google Colab and a local Jupyter kernel (for example VS Code). Cells that differ between the two detect the environment on their own.

## Installing and Importing the Necessary Libraries

In [1]:
# Install all required packages.
# In Google Colab this uses uv for fast dependency resolution (as in the course template).
# In a local kernel (VS Code / Jupyter) it uses pip for the active kernel, and also
# installs sentence-transformers and openpyxl, which Colab already ships with.
import shutil
import subprocess
import sys

PACKAGES = [
    "docling==2.126.0",
    "pandas==2.2.3",
    "transformers==5.16.1",
    "langchain-huggingface==1.2.2",
    "langchain-text-splitters==1.1.2",
    "langchain-community==0.4.2",
    "faiss-cpu==1.15.0",
    "langchain-openai==1.6.1",
    "deepeval==4.2.2",
    "PyGithub==2.6.1",
]

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and shutil.which("uv"):
    install_cmd = ["uv", "pip", "install", "-q", *PACKAGES]
else:
    install_cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        *PACKAGES, "sentence-transformers==6.1.0", "openpyxl",
    ]

completed = subprocess.run(install_cmd, capture_output=True, text=True)
print(completed.stdout[-2000:])
if completed.returncode != 0:
    print(completed.stderr[-4000:])
    raise RuntimeError("Package installation failed. See the message above.")
print("Installation finished. Restart the runtime/kernel, then continue from the next cell.")


Installation finished. Restart the runtime/kernel, then continue from the next cell.


**Note:**

- After running the cell above, restart the runtime (Google Colab) or the notebook kernel (VS Code / Jupyter), then run all cells from the next cell onward.
- A warning about package dependencies can be ignored. The pinned versions above are the ones this notebook needs.

In [2]:
# Import function for displaying outputs in Jupyter/Google Colab
from IPython.display import display, Markdown

# Import library for working with tabular data
import pandas as pd

# To suppress warnings
import warnings

# To make API requests
import requests

# Import Docling's document converter for processing PDF documents
from docling.document_converter import DocumentConverter

# Import tokenizer for determining token counts and preparing text for chunking
from transformers import AutoTokenizer

# Import LangChain's recursive character text splitter for token-aware chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import libraries for working with JSON data and OS related functionality
import json
import os

# Import LangChain's Document class for representing text and metadata
from langchain_core.documents import Document as LCDocument

# Import Hugging Face embedding model integration for creating vector embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Import OpenAI-compatible chat model for generating RAG responses
from langchain_openai import ChatOpenAI

# Import FAISS vector database for storing and retrieving document embeddings
from langchain_community.vectorstores import FAISS

# Import DeepEval evaluation framework
from deepeval import evaluate

# Import DeepEval test case class for defining evaluation inputs and outputs
from deepeval.test_case import LLMTestCase

# Import DeepEval metrics for evaluating different aspects of the RAG system
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval

# Import OpenAI model wrapper used as the evaluation/judge model in DeepEval
from deepeval.models import OpenAIModel

# Detect whether the notebook is running in Google Colab (used for the GitHub token later)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Running in Google Colab:", IN_COLAB)

C:\Users\skcku\AppData\Local\Temp\ipykernel_31724\2325052993.py:36: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Running in Google Colab: False


All the key libraries are imported here. The main components are:

- **Docling** for parsing PDF documents into structured markdown.
- **LangChain** modules for text splitting, embeddings, vector storage, and LLM interaction.
- **DeepEval** for evaluating the quality of the RAG pipeline using LLM-as-a-Judge metrics.
- Pandas display options are set to show full row/column content without truncation, which helps when inspecting long text chunks.
- DeprecationWarnings are suppressed to keep the notebook output clean.

## Data Understanding

### Loading the documents

- We define the file paths to the four hotel policy PDF documents that form the knowledge base for this RAG system.
- Each document covers a different area of hotel operations.

In [3]:
# Create the root folder
root_path = 'Data'

os.makedirs(root_path, exist_ok=True)

**Note:** After executing the previous cell, a folder named `Data` will be created. Ensure that all the PDF files, `evaluation.csv`, and the test file are uploaded to this folder before executing the code cells below.

In [4]:
# List all policy document PDF file paths
pdf_paths = [
    root_path + "/" + "aurorastay_room_and_property_information.pdf",
    root_path + "/" + "aurorastay_refunds_and_additional_charges_guide.pdf",
    root_path + "/" + "aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf",
    root_path + "/" + "aurorastay_booking_terms_and_cancellation_policy.pdf"
]

In [5]:
# Pre-flight check: confirm every input file is in place before the slow steps start.
# This avoids discovering a missing file after a 10 to 15 minute PDF conversion.
required_files = pdf_paths + [root_path + "/evaluation.csv", "config.json"]
missing = [p for p in required_files if not os.path.exists(p)]

# The test set is accepted as either CSV or Excel
if not any(os.path.exists(root_path + "/test" + ext) for ext in (".csv", ".xlsx")):
    missing.append(root_path + "/test.csv (or test.xlsx)")

if missing:
    raise FileNotFoundError(
        "Upload these files before continuing:\n  - " + "\n  - ".join(missing)
    )
print("All required files found.")

FileNotFoundError: Upload these files before continuing:
  - Data/aurorastay_room_and_property_information.pdf
  - Data/aurorastay_refunds_and_additional_charges_guide.pdf
  - Data/aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf
  - Data/aurorastay_booking_terms_and_cancellation_policy.pdf
  - Data/evaluation.csv
  - config.json
  - Data/test.csv (or test.xlsx)

Four policy documents are loaded:

1. **Room and property information** covering room types, amenities, and property-specific details.
2. **Refunds and additional charges** explaining fee structures and refund eligibility.
3. **Guest conduct, accessibility, and resolution guidelines** for behavioral policies and dispute handling.
4. **Booking terms and cancellation policy** with reservation rules and cancellation windows.

Together, these documents should cover the majority of questions hotel staff receive from guests.

- The `DocumentConverter` from Docling handles PDF parsing.
- It extracts text, tables, and document structure from PDFs and converts them into a format that preserves headings, lists, and table layouts.
- This matters because hotel policy documents contain structured tables (for example fee schedules and the property matrix) that would be lost with a simple text extractor.

In [ ]:
# Initialize the Docling document converter
converter = DocumentConverter()

- Each PDF is converted one at a time and stored in a dictionary keyed by its file path.
- The converter processes the document layout, extracts text and tables, and returns a Docling document object.
- The print statement after each file confirms that the conversion completed successfully.

**Note:** The code cell below may take 10 to 15 minutes to complete.

In [ ]:
# Convert each policy PDF into a Docling document object
docs = {}
for path in pdf_paths:
    result = converter.convert(path)
    docs[path] = result.document
    print(path, " Completed ", end='\n')

### Viewing the documents

#### Document structure summary

Before reading the raw text, a quick profile of each converted document: page count, size, number of headings, and number of tables. This is the basis for the chunking decisions made later.

In [ ]:
# Profile each converted document: pages, length, headings, and tables
structure_rows = []
for path, doc in docs.items():
    markdown_text = doc.export_to_markdown()
    lines = markdown_text.splitlines()

    structure_rows.append({
        "Document": os.path.basename(path),
        "Pages": len(getattr(doc, "pages", {}) or {}),
        "Characters": len(markdown_text),
        "Words": len(markdown_text.split()),
        # Markdown headings start with '#'
        "Headings": sum(1 for line in lines if line.lstrip().startswith("#")),
        "Tables": len(getattr(doc, "tables", []) or []),
        # Markdown table rows start with '|'
        "Table rows": sum(1 for line in lines if line.lstrip().startswith("|")),
    })

structure_df = pd.DataFrame(structure_rows)
display(structure_df)

In [ ]:
# List the section headings of each document to see how the content is organized
for path, doc in docs.items():
    print(f"\n=== {os.path.basename(path)} ===")
    for line in doc.export_to_markdown().splitlines():
        if line.lstrip().startswith("#"):
            print("  ", line.strip())

#### Viewing the markdown

In [ ]:
# Select one document to preview its converted markdown output
path = root_path + "/" + "aurorastay_refunds_and_additional_charges_guide.pdf"

# Display the converted document as markdown to inspect its preserved structure
print(docs[path].export_to_markdown())

- The markdown export shows how Docling preserved the original document structure.
- Headings, subheadings, bullet points, and tables are all retained in markdown format.
- This is the text that will later be split into chunks for the vector database, so clean, well-structured markdown directly improves retrieval quality.

#### Viewing the tables

Tables are particularly important in hotel policy documents because they hold structured data such as fee schedules, room configurations, and property-specific details.

In [ ]:
# Extract and display all tables from the converted documents
for path, doc in docs.items():
    for item, _ in doc.iterate_items():
        if getattr(item, "label", "") == "table":
            print(f"\n--- Table from: {path} ---")
            print(item.export_to_markdown(doc=doc))

- The loop iterates through every item in each document and filters for items labeled as "table".
- Each table is exported as markdown, preserving its row/column structure.
- This confirms that Docling extracted the tabular data from the PDFs, which is essential since many guest queries involve looking up specific values from these tables.

### Observations on document structure and details

- **Size and scope.** The knowledge base is small: four PDFs and 19 pages in total (5, 5, 4, and 5 pages). Each document owns one policy area, so the source file name is useful metadata for tracing an answer back to its policy.
- **Structure.** Docling exports each document as markdown with section headings, body text, lists, and tables (see the heading listing above). Because the content is organized by policy section, the chunker below splits at headings first, so that a chunk tends to hold one complete rule instead of fragments of two.
- **Tables carry the specifics.** Amounts, deadlines, and per-property differences sit in tables: the six-property quick-reference matrix in the booking terms, the itemized additional charges in the refunds guide, and the escalation matrix in the conduct guidelines. A table row has little meaning without its header row, so chunks should be large enough to keep a table together where possible.
- **Topics span documents.** Several topics are split across files. Pets appear as a policy in the room and property document and as a fee in the refunds guide. Service animals appear in both the room document and the conduct guidelines. Cancellation rules sit in the booking terms while the resulting refund sits in the refunds guide. This is the reason around 40% of staff queries need more than one document, and it means retrieval must return more than one or two chunks for those queries.
- **Conditions matter.** Most rules are conditional on rate plan (Flexible, Advance Purchase, member/promotional, group) or on property. An answer that omits the condition is incomplete, which shapes the generation rules below.
- **Procedures are ordered.** The five-step refund dispute workflow and the five-stage complaint resolution process are sequences. Splitting one across chunks risks returning only part of the procedure.

*The counts in the structure table above (headings, tables, words) come from this run and should be read alongside these points.*

## Retrieval Augmented Generation System

### Chunking

#### Choosing the embedding model

The embedding model chosen is **"HarishMaths/Hotel-Policy-Embedding"**, which is fine-tuned on hotel policy documents. It is expected to provide better retrieval quality on this domain than a general-purpose base model.

In [ ]:
# Define the model name
EMBED_MODEL = "HarishMaths/Hotel-Policy-Embedding"

# Load the domain-specific embedding model tokenizer and check its maximum token limit
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

print(f"Model max tokens: {tokenizer.model_max_length}")

- The model's maximum token length is the upper bound on how much text can be encoded into a single embedding vector.
- This limit directly affects the chunking strategy in the next step.
- Chunks that exceed this limit are silently truncated, which loses information.

Before chunking, it helps to see how the tokenizer breaks text into tokens. This gives a sense of the token-to-word ratio for hotel policy language.

In [ ]:
# A realistic staff query to inspect tokenization
sample = "Can a guest on an Advance Purchase rate get a refund if they cancel two days before arrival?"

tokens = tokenizer.tokenize(sample)
print(f"Sample: '{sample}'")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"Tokens per word: {len(tokens) / len(sample.split()):.2f}")

- The sample sentence produces a specific number of tokens, and we can see how individual words are split into subwords.
- Knowing this ratio helps when reasoning about chunk sizes: a chunk of a few hundred tokens holds a complete policy section in most cases.

#### Define the chunker

**Design choices**

- **Token-aware splitting.** Chunk length is measured with the embedding model's own tokenizer, so a chunk can never be longer than what the model can encode.
- **Chunk size is derived from the model limit**, at about 80% of it (capped at 400 tokens). The headroom covers the special tokens the model adds and avoids silent truncation.
- **Overlap of about 15%** keeps a rule and its exception together when a boundary falls between them.
- **Separators follow the markdown structure**: split at `##` and `###` headings first, then paragraphs, then lines, then sentences. This keeps each chunk aligned to a policy section instead of cutting mid-sentence.

In [ ]:
# Some tokenizers report a very large placeholder value when no limit is set,
# so cap the usable limit at 512 tokens.
MODEL_MAX_TOKENS = min(tokenizer.model_max_length, 512)

# Leave ~20% headroom below the model limit for special tokens; never exceed 400 tokens
CHUNK_SIZE = min(400, int(MODEL_MAX_TOKENS * 0.8))

# ~15% overlap so that a condition and its exception are not separated at a boundary
CHUNK_OVERLAP = int(CHUNK_SIZE * 0.15)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=CHUNK_SIZE,          # measured in tokens of the embedding model
    chunk_overlap=CHUNK_OVERLAP,
    # Prefer section boundaries, then paragraphs, lines, sentences, words
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
)

print(f"Chunk size: {CHUNK_SIZE} tokens | Overlap: {CHUNK_OVERLAP} tokens")

#### Chunking the documents

In [ ]:
# Wrap each document's Markdown export as a single LangChain Document, then
# split it into chunks using the RecursiveCharacterTextSplitter
raw_docs = [
    LCDocument(page_content=doc.export_to_markdown(), metadata={"source": path})
    for path, doc in docs.items()
]

lc_docs = splitter.split_documents(raw_docs)

print(f"Total chunks: {len(lc_docs)}")

- Each document's markdown export is first wrapped into a single LangChain `Document` with its source path as metadata. The splitter then breaks these into smaller chunks.
- The total chunk count is the number of vectors stored in the database. Each chunk keeps the source metadata, so any retrieved chunk can be traced back to its original document.

In [ ]:
# Chunk statistics: how many chunks each document produced and how long they are
chunk_stats = pd.DataFrame({
    "Document": [os.path.basename(c.metadata["source"]) for c in lc_docs],
    "Tokens": [len(tokenizer.tokenize(c.page_content)) for c in lc_docs],
})

display(
    chunk_stats.groupby("Document")["Tokens"]
    .agg(Chunks="count", Min="min", Mean="mean", Max="max")
    .round(1)
)
print(f"Longest chunk: {chunk_stats['Tokens'].max()} tokens (limit used: {MODEL_MAX_TOKENS})")

Let us preview the first 400 characters of each chunk to verify the split looks reasonable.

In [ ]:
# Preview the text content of each generated chunk
for i, chunk in enumerate(lc_docs):
    print(f"\n--- Chunk {i} | {chunk.metadata['source']} ---")
    print(chunk.page_content[:400])
    print("..." if len(chunk.page_content) > 400 else "")

- Scanning the previews shows whether splits happen at logical boundaries.
- Good chunks start and end at natural breakpoints such as section headings or paragraph boundaries, not in the middle of a sentence.
- The overlap between consecutive chunks should also be visible.
- For chunks that contain a table, check that the header row is in the same chunk as its data rows.

Let us also verify the metadata attached to each chunk.

In [ ]:
# Preview the metadata of each generated chunk
for i, chunk in enumerate(lc_docs):
    print(f"\n--- Chunk {i} ---")
    print(chunk.metadata)

- Each chunk carries a `source` field pointing to the PDF it came from.
- This metadata is preserved through the vector store and returned alongside retrieved chunks during inference, which allows the final answer to cite its source document.

### Vector Database

#### Loading the embedding model

In [ ]:
# Load the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,                         # Pre-trained embedding model
    model_kwargs={"device": "cpu"},                 # Run the model on CPU
    encode_kwargs={"normalize_embeddings": True},   # Normalize embeddings for similarity search
)

- The embedding model is loaded onto the CPU and configured to normalize its output vectors.
- Normalization makes cosine similarity and inner product give the same ranking, which keeps the FAISS similarity search consistent.
- Because this model is fine-tuned on hotel policy data, it should produce better embeddings for this use case than a general-purpose model.

The embedding model can enforce a shorter sequence limit than the tokenizer reports. The check below confirms that no chunk will be truncated when it is embedded.

In [ ]:
# Safety check: compare chunk lengths against the limit the embedding model actually enforces
encoder = getattr(embeddings, "_client", None) or getattr(embeddings, "client", None)
encoder_limit = getattr(encoder, "max_seq_length", None)

if encoder_limit:
    # Allow 2 tokens for the special start/end tokens the model adds
    too_long = int((chunk_stats["Tokens"] + 2 > encoder_limit).sum())
    print(f"Embedding model sequence limit: {encoder_limit} tokens")
    print(f"Chunks that would be truncated: {too_long} of {len(chunk_stats)}")
    if too_long:
        print(
            "WARNING: lower CHUNK_SIZE to about "
            f"{int(encoder_limit * 0.8)} tokens and re-run the chunking cells."
        )
else:
    print("Could not read the embedding model's sequence limit; relying on the tokenizer limit.")

#### Creating the Vector Database

FAISS is used as the vector store. With a knowledge base of this size an exact (flat) index is fast and needs no tuning, and the index can be saved to disk for deployment.

In [ ]:
# Build the FAISS vector store from the chunks using the same embedding model
vectorstore = FAISS.from_documents(lc_docs, embeddings)

print(f"Vectors stored: {vectorstore.index.ntotal}")

#### Retrieval System

The retrieval function wraps FAISS similarity search into a clean interface. Given a query string, it returns the top-k most similar chunks along with their metadata.

In [ ]:
# Retrieve the top-k relevant chunks from the vector store
def retrieve(query: str, k: int = 2, vectorstore=None) -> list[str]:
    results = vectorstore.similarity_search(query, k=k)

    # Return each chunk with its content and metadata
    return [
        f"Content:\n{r.page_content}\n\nMetadata:\n{r.metadata}"
        for r in results
    ]

- `retrieve` performs a similarity search on the vector store and returns formatted strings containing both the chunk content and its metadata.
- Formatting the results this way makes it easy to pass them directly into the LLM prompt later.
- The `k` parameter controls how many chunks are retrieved, defaulting to 2.

Let us test the retrieval system with a sample query about pet charges. (To make the query property-specific, add a property name from the quick-reference matrix printed in the tables above.)

In [ ]:
# A sample query to test retrieval
query = "What is the pet fee, and are there any restrictions on bringing pets?"

for i, chunk in enumerate(retrieve(query, vectorstore=vectorstore)):
    print(f"\n--- Result {i} ---")
    print(chunk)

- The retrieved chunks should contain information about pet policies and charges.
- If the chunks are relevant to the query, the embedding model and vector store are working together correctly.
- Pet information is split between the room and property document (policy) and the refunds guide (fee), so a complete answer may need chunks from both. This is an early sign that `k` larger than 2 will help on multi-document questions.

### Generation System

#### Loading the LLM

We load the OpenAI API credentials from `config.json`.

**Note:** Before executing the cell below, ensure that `config.json` is uploaded next to the notebook.

In [ ]:
# Load API configuration from the JSON config file
with open("config.json", "r") as f:
    config = json.load(f)

# Set environment variables for OpenAI API access
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]
os.environ["OPENAI_API_BASE"] = config["OPENAI_API_BASE"]
# The OpenAI SDK reads this name for the base URL; setting both keeps every client consistent
os.environ["OPENAI_BASE_URL"] = config["OPENAI_API_BASE"]

print("Configuration loaded.")

- The API key and base URL are loaded from a local `config.json` file and set as environment variables.
- This keeps credentials out of the notebook code.
- The same environment variables are picked up by LangChain's `ChatOpenAI` wrapper automatically.

**Generation settings**

- **Model:** `gpt-4o-mini`. The task is to restate retrieved policy text accurately, which does not need a larger model, and a smaller model keeps the cost per query low.
- **Temperature 0:** policy answers should be the same every time the same question is asked.
- **Top-p 1:** with temperature at 0 there is no reason to restrict the token distribution further.
- **Max tokens 512:** enough for an answer that lists conditions for several rate plans or properties without being cut off.

In [ ]:
# Generation model and sampling parameters
MODEL = "gpt-4o-mini"   # Generator LLM
TEMPERATURE = 0         # Deterministic, consistent policy answers
TOP_P = 1               # No extra restriction on token selection
MAX_TOKENS = 512        # Room for multi-condition answers

# Initialize the LLM
llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_API_BASE"],
)

In [ ]:
# A connectivity check for the LLM
TEST_QUERY_LLM = "Reply with exactly: LLM is ready"

# Send the test query to the LLM
response = llm.invoke(TEST_QUERY_LLM)

# Display the LLM response
print(response.content)

- A quick connectivity check to make sure the LLM is reachable and responding.
- If the output says "LLM is ready", the API key, base URL, and model name are all configured correctly.

#### Defining the generation system

The rules below map to the four risks in the problem statement:

| Rule | Risk it addresses |
|---|---|
| 1, 5, 8, 9 | Faithfulness: only retrieved facts, exact figures, explicit fallback, surface conflicts |
| 3, 4 | Completeness: keep the conditions (rate plan, property, deadlines, exceptions) that change the answer |
| 2, 6, 7 | Answer relevancy: answer first, leave out unrelated retrieved text, stay concise |

In [ ]:
# Define the generation function that prompts the LLM with the retrieved context
def generate(query, retrieved_chunks, model) -> str:

    rules = """
    1. Answer only from the Retrieved Policy Information. Do not use outside knowledge or assumptions.
    2. Answer the question that was asked in the first sentence.
    3. Include every condition that changes the answer: rate plan, property, deadlines, fees, and exceptions.
    4. If the answer differs by property or rate plan and the question does not specify one, state each case.
    5. Give amounts, times, and deadlines exactly as written in the policy. Do not round or estimate.
    6. Leave out retrieved information that does not bear on the question.
    7. Keep the answer concise, clear, and easy for hotel staff to relay to a guest.
    8. If the retrieved information does not contain the answer, reply exactly:
       "The available policy documents do not contain enough information to answer this question."
    9. If retrieved passages conflict with each other, state the conflict instead of choosing one.
    """  # Rules for how the LLM should generate the answer

    prompt = f"""
    You are an AI assistant answering questions about hotel policies.

    User Query:
    {query}

    Retrieved Policy Information:
    {retrieved_chunks}

    Rules:
    {rules}

    Answer:
    """

    return model.invoke(prompt)

#### Defining the RAG Function

- The `rag` function ties the retrieval and generation steps together into a single callable pipeline.
- It creates an LLM instance with the specified parameters, retrieves the top-k chunks, and passes them to the generation function.
- It returns both the generated answer and the retrieved chunks, which is useful for evaluation and for displaying source information in the UI.

In [ ]:
# Define the full RAG pipeline combining retrieval and generation
def rag(
    query: str,
    k: int = 2,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = 512,
    vectorstore=None
):
    # Create the generator model (credentials are read from the environment variables set above)
    model = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )

    # Retrieve relevant chunks
    retrieved_chunks = retrieve(query=query, k=k, vectorstore=vectorstore)

    # Generate answer using retrieved chunks
    answer = generate(
        query=query,
        retrieved_chunks=retrieved_chunks,
        model=model
    )

    return answer.content, retrieved_chunks

Let us run the full RAG pipeline end to end with the same pet charge query to see the complete flow.

In [ ]:
# Run the full pipeline on a sample query
query = "What is the pet fee, and are there any restrictions on bringing pets?"

response, rel_chunks = rag(query, vectorstore=vectorstore)

# Print the generated response
print("Response: ", response)

# Print the sources the answer was grounded on
print("\nSources used:")
for i, chunk in enumerate(rel_chunks, start=1):
    print(f"  Chunk {i}: {chunk.split('Metadata:')[-1].strip()}")

The grounding rules can be checked with a question the policy documents do not cover. The expected behavior is the fallback sentence from rule 8, not an invented answer.

In [ ]:
# An out-of-scope question to confirm the model declines instead of guessing
out_of_scope_query = "What is the nightly rate for a helicopter transfer from the airport?"

response, _ = rag(out_of_scope_query, vectorstore=vectorstore)
print("Response: ", response)

## Evaluation Pipeline

### Loading the Evaluation Data

- The evaluation dataset contains pre-written questions, expected answers, and the relevant context that a correct answer should be based on.
- This lets us measure how well the RAG pipeline performs in a repeatable way.

**Note:** Before executing the cells below, ensure that `evaluation.csv` and the test file (`test.csv` or `test.xlsx`) are uploaded to the `Data` folder.

In [ ]:
# Helper to load a question/answer file from the Data folder.
# The test set is distributed as either CSV or Excel, so both are supported.
REQUIRED_COLUMNS = ["question", "expected_answer", "relevant_context"]


def load_qa_file(stem: str) -> pd.DataFrame:
    csv_path = f"{root_path}/{stem}.csv"
    xlsx_path = f"{root_path}/{stem}.xlsx"

    if os.path.exists(csv_path):
        try:
            data = pd.read_csv(csv_path)
        except UnicodeDecodeError:
            # Files saved from Excel on Windows are often not UTF-8
            data = pd.read_csv(csv_path, encoding="cp1252")
    elif os.path.exists(xlsx_path):
        data = pd.read_excel(xlsx_path)
    else:
        raise FileNotFoundError(f"Neither {csv_path} nor {xlsx_path} was found.")

    # Tidy the column names and confirm the expected columns are present
    data.columns = [str(c).strip() for c in data.columns]
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in data.columns]
    if missing_cols:
        raise ValueError(
            f"{stem} is missing columns {missing_cols}. Columns found: {list(data.columns)}"
        )

    # Empty cells would otherwise break the string handling in the test case builder
    for col in REQUIRED_COLUMNS:
        data[col] = data[col].fillna("").astype(str)

    return data

In [ ]:
# Load evaluation data
df = load_qa_file("evaluation")

print(f"Evaluation queries: {len(df)}")
display(df.head())

- Each row is one test case with columns for the question, the expected answer, and the relevant context passages.
- These are compared against the RAG pipeline's actual outputs during the evaluation step.

### Initializing the Evaluation Model

For evaluation we use a separate, stronger model (`gpt-4o`) as the judge. Using a different model for evaluation avoids the bias that comes from having the same model grade its own answers.

In [ ]:
# Initialize the evaluation model using a stronger LLM for scoring
EVAL_MODEL = 'gpt-4o'

eval_model = OpenAIModel(
    model=EVAL_MODEL,
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_API_BASE"]
)

In [ ]:
# Verify the evaluation model is responding correctly
eval_model.generate('Hi')

The evaluation model is responding correctly.

### Defining Evaluation Metrics

Four metrics are defined, each corresponding to one of the risks in the problem statement:

- **Faithfulness** guards against unsupported information.
- **Contextual Recall** checks retrieval completeness.
- **Contextual Precision** checks that retrieved chunks are relevant.
- **Answer Relevancy** checks that the response addresses the question asked.

In [ ]:
# Faithfulness
# Checks whether the generated answer is factually supported by
# the information retrieved from the knowledge base.
faithfulness_metric = GEval(
    name="Faithfulness",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the actual output is factually supported by "
        "the provided retrieval context."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # First, identify the individual factual claims made in the answer.
        "Break the actual output into individual factual claims.",

        # Check each claim against the retrieved context.
        "Check whether each claim is supported by the retrieval context.",

        # Lower the score if the answer contains unsupported or conflicting claims.
        "Penalize claims that are unsupported or contradicted by the retrieval context.",

        # Differences in wording should not affect the score if the meaning is supported.
        "Do not penalize differences in wording when the meaning is supported by the context.",

        # Give a high score when the important claims are supported.
        "Give a higher score when all important claims are supported by the context."
    ],

    # Specifies which parts of the test case are used for evaluation.
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.85,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Contextual Recall
# Checks whether the retrieved context contains the information
# needed to produce the expected answer.
recall_metric = GEval(
    name="Contextual Recall",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the retrieval context contains the information "
        "needed to support the expected output."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Identify the important claims that the expected answer should contain.
        "Break the expected output into its important factual claims.",

        # Check whether those claims can be found or supported by the retrieved context.
        "Check whether each important claim is present or supported by the retrieval context.",

        # Lower the score when important information is missing from retrieval.
        "Penalize the retrieval context when important information required for the expected output is missing.",

        # Wording differences should not affect the evaluation.
        "Do not penalize differences in wording or presentation.",

        # Give a high score when retrieval contains most or all required information.
        "Give a higher score when the retrieval context contains most or all information needed to produce the expected output."
    ],

    # Recall requires the expected answer and retrieved context.
    evaluation_params=[
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.80,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Contextual Precision
# Checks whether the retrieved context is relevant to the user's question
# and whether relevant information is prioritized over irrelevant information.
precision_metric = GEval(
    name="Contextual Precision",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the retrieved context is relevant to answering "
        "the input question, with relevant information prioritized over irrelevant information."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Identify which parts of the retrieved context are useful for the question.
        "Identify the information in the retrieval context that is relevant to the input question.",

        # Check whether unnecessary or unrelated information was retrieved.
        "Check whether the retrieval context contains unnecessary or unrelated information.",

        # Lower the score when a large portion of the retrieved context is irrelevant.
        "Penalize retrievals that contain a large amount of irrelevant information.",

        # Give more credit when useful information appears prominently.
        "Give more credit when relevant information is prioritized.",

        # Give a high score when most retrieved information is useful.
        "Give a higher score when most of the retrieved context is useful for answering the question."
    ],

    # Precision evaluates the question against the retrieved context.
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.75,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Answer Relevancy
# Checks whether the generated answer directly addresses the user's question
# without including unnecessary or unrelated information.
answer_relevancy_metric = GEval(
    name="Answer Relevancy",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the actual output directly addresses the input "
        "question and provides an appropriate response without unnecessary information."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Check whether the answer directly responds to the question.
        "Check whether the actual output directly answers the input question.",

        # Check whether the information provided is relevant to the question.
        "Check whether the response contains information relevant to the question.",

        # Lower the score for unrelated or unnecessary information.
        "Penalize unnecessary, unrelated, or off-topic information.",

        # A short answer should not be penalized if it answers the question sufficiently.
        "Do not penalize concise answers when they adequately address the question.",

        # Give a high score when the answer is focused and directly relevant.
        "Give a higher score when the response is focused and directly relevant to the question."
    ],

    # Answer relevancy evaluates the question against the generated answer.
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.75,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Store all evaluation metrics in a single list.
# This list can then be passed to DeepEval's evaluation function.
metrics = [
    faithfulness_metric,
    recall_metric,
    precision_metric,
    answer_relevancy_metric
]

Four evaluation metrics are defined, each targeting a specific aspect of RAG quality:

- **Faithfulness (threshold >= 0.85):** checks whether the generated answer is factually supported by the retrieved context. Uses `ACTUAL_OUTPUT` and `RETRIEVAL_CONTEXT`.
- **Contextual Recall (threshold >= 0.80):** checks whether the retrieved context contains the information needed to produce the expected answer. Uses `EXPECTED_OUTPUT` and `RETRIEVAL_CONTEXT`.
- **Contextual Precision (threshold >= 0.75):** checks whether the retrieved chunks are relevant to the question and whether useful information is prioritized. Uses `INPUT` and `RETRIEVAL_CONTEXT`.
- **Answer Relevancy (threshold >= 0.75):** checks whether the answer directly addresses the user's question without unnecessary information. Uses `INPUT` and `ACTUAL_OUTPUT`.

All four use the GEval framework, which gives the judge model step-by-step evaluation instructions. The thresholds match the success criteria in the problem statement.

### Building Evaluation Test Cases

The function below takes the evaluation DataFrame and runs the RAG pipeline on each question. It then packages the inputs, outputs, and ground-truth data into DeepEval test cases for scoring.

In [ ]:
def test_case_generator(df, vectorstore, k=3, temperature=0, top_p=1, max_tokens=512):

    # Store the generated DeepEval test cases
    test_cases = []

    # Process each evaluation question
    for _, row in df.iterrows():

        # Run the RAG pipeline
        answer, retrieved_chunks = rag(
            query=row["question"],
            k=k,
            model_name=MODEL,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            vectorstore=vectorstore
        )

        # Split the ground-truth relevant context
        # Multiple contexts are separated by "<>"
        relevant_context = [
            c.strip()
            for c in row["relevant_context"].split("<>")
            if c.strip()
        ]

        # Create the DeepEval test case
        test_case = LLMTestCase(
            # User question
            input=row["question"],

            # Answer generated by the RAG pipeline
            actual_output=answer,

            # Ground-truth expected answer
            expected_output=row["expected_answer"],

            # Context actually retrieved by the RAG pipeline
            retrieval_context=retrieved_chunks,

            # Ground-truth relevant context
            context=relevant_context
        )

        # Add test case
        test_cases.append(test_case)

    return test_cases

For each row in the evaluation data, the function:

1. Runs the full RAG pipeline (retrieve + generate) on the question.
2. Splits the ground-truth relevant context on the `<>` delimiter, since some questions need multiple context passages.
3. Builds an `LLMTestCase` containing the input question, the generated answer, the expected answer, the retrieved context, and the ground-truth context.

This gives DeepEval everything it needs to score each metric.

### Running the Evaluation

**Choice of parameters**

- **K = 4.** Around 40% of queries need information from more than one document, so retrieving only 2 chunks risks missing part of the answer (lower Contextual Recall). Retrieving many more adds unrelated text (lower Contextual Precision). Four chunks is the starting balance; re-run this cell with 3 and 5 to compare.
- **Temperature 0, Top-p 1, Max tokens 512**, matching the generation settings chosen earlier, so the evaluation measures the configuration that is deployed.

The step below can take a few minutes: it runs the RAG pipeline on every question and then scores each answer on four metrics with the judge model.

In [ ]:
# Define the parameters for test case generation
K = 4               # Number of chunks to retrieve for each query
TEMPERATURE = 0     # Deterministic answers
TOP_P = 1           # No extra restriction on token selection
MAX_TOKENS = 512    # Maximum number of tokens for the response

# Create the test cases
test_cases = test_case_generator(
    df,
    vectorstore,
    k=K,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)

# Run evaluation
results = evaluate(
    test_cases=test_cases,
    metrics=metrics
)

**Note:** The results above come from an LLM-as-a-Judge approach. LLM-based evaluation is not fully deterministic, so the metric scores can vary slightly between runs. A single run should not be treated as definitive; running the evaluation several times and looking at the overall trend gives a more reliable estimate.

**Note:** If the results are not satisfactory, re-run the previous cell with different parameter values such as `K`. Once the results are satisfactory, note down the values used.

The function below converts raw evaluation results into a structured DataFrame with composite scores and routing decisions.

In [ ]:
def create_evaluation_df(test_cases, results):

    rows = []

    # Loop through each test case and its corresponding evaluation result
    for test_case, result in zip(test_cases, results.test_results):

        # Create a dictionary of metric name -> score
        # Remove "[GEval]" from the metric names so they match our keys
        scores = {
            metric.name.replace(" [GEval]", "").strip().lower(): metric.score
            for metric in result.metrics_data
        }

        # Calculate the weighted composite score
        composite = (
            0.35 * scores["faithfulness"]
            + 0.25 * scores["contextual recall"]
            + 0.20 * scores["contextual precision"]
            + 0.20 * scores["answer relevancy"]
        )

        # Determine the routing type based on the composite score
        if composite > 0.75:
            routing_type = "AUTO ANSWER"
        elif composite >= 0.50:
            routing_type = "ASSISTED REVIEW"
        else:
            routing_type = "MANUAL LOOKUP"

        # Store the evaluation results for this question.
        # The question is read from the result itself so that the label always
        # belongs to the scores on the same row, whatever order results come back in.
        rows.append({
            "Question": getattr(result, "input", None) or test_case.input,
            "Faithfulness Score": scores["faithfulness"],
            "Contextual Recall Score": scores["contextual recall"],
            "Contextual Precision Score": scores["contextual precision"],
            "Answer Relevancy Score": scores["answer relevancy"],
            "Composite Score": composite,
            "Type": routing_type
        })

    return pd.DataFrame(rows)

The composite score uses the weights from the success criteria: Faithfulness has the highest weight (35%) because avoiding unsupported information is the top priority. Contextual Recall (25%) is next, followed by Contextual Precision (20%) and Answer Relevancy (20%).

The table labels each query with a routing type:

- Above 0.75: **AUTO ANSWER**, reliable enough to use directly.
- 0.50 to 0.75: **ASSISTED REVIEW**, a useful draft that needs a human check.
- Below 0.50: **MANUAL LOOKUP**.

The problem statement defines two actions only (Auto Answer at a composite of 0.75 or more, otherwise Manual Lookup). The success criterion "Queries Confidently Answered" is therefore measured below as the share of queries with a composite score of at least 0.75. The middle band is kept in the table as extra detail on how far a query is from the cutoff; under the problem statement it counts as Manual Lookup.

In [ ]:
# Generate the evaluation results DataFrame with composite scores and routing decisions
eval_results_df = create_evaluation_df(test_cases, results)
display(eval_results_df)

In [ ]:
# Compare the results against the success criteria from the problem statement
THRESHOLDS = {
    "Faithfulness Score": 0.85,
    "Contextual Recall Score": 0.80,
    "Contextual Precision Score": 0.75,
    "Answer Relevancy Score": 0.75,
}
AUTO_ANSWER_CUTOFF = 0.75      # Composite score needed for Auto Answer
AUTO_ANSWER_TARGET = 0.50      # Share of queries that should reach Auto Answer


def summarize_results(result_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    # Average of each metric across all queries, compared with its threshold
    for column, threshold in THRESHOLDS.items():
        achieved = result_df[column].mean()
        rows.append({
            "Success Criterion": column.replace(" Score", ""),
            "Achieved": round(achieved, 3),
            "Threshold": f">= {threshold:.2f}",
            "Met": "Yes" if achieved >= threshold else "No",
        })

    # Share of queries whose composite score reaches the Auto Answer cutoff
    auto_share = (result_df["Composite Score"] >= AUTO_ANSWER_CUTOFF).mean()
    rows.append({
        "Success Criterion": "Queries Confidently Answered",
        "Achieved": round(auto_share, 3),
        "Threshold": f">= {AUTO_ANSWER_TARGET:.2f}",
        "Met": "Yes" if auto_share >= AUTO_ANSWER_TARGET else "No",
    })

    return pd.DataFrame(rows)


eval_summary_df = summarize_results(eval_results_df)
display(eval_summary_df)
print(f"Mean composite score: {eval_results_df['Composite Score'].mean():.3f}")

- The first table shows each evaluation question with its individual metric scores, the weighted composite score, and the routing decision.
- The second table compares the averages with the success criteria, which shows at a glance where the system meets the bar and where it does not.
- Queries at or above a composite of 0.75 are the ones the system can handle without manual lookup.

## Evaluating on the Test set

### Loading the Test Data

We now load a separate test dataset to evaluate the system on questions it has not been tuned against. This gives a more honest estimate of real-world performance.

In [ ]:
# Load test data (test.csv or test.xlsx)
df = load_qa_file("test")

print(f"Test queries: {len(df)}")
display(df.head())

### Running the Evaluation

The same parameters chosen on the evaluation set are used here without further tuning.

In [ ]:
# Create the test cases and run evaluation
test_cases = test_case_generator(
    df,
    vectorstore,
    k=K,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)
results = evaluate(
    test_cases=test_cases,
    metrics=metrics
)

The same evaluation pipeline is applied to the test set. The RAG pipeline first generates answers for all test questions, then DeepEval scores each one across the four metrics. This can take several minutes.

In [ ]:
# Display the test set evaluation results with composite scores and routing decisions
test_results_df = create_evaluation_df(test_cases, results)
display(test_results_df)

In [ ]:
# Compare the test results against the success criteria
test_summary_df = summarize_results(test_results_df)
display(test_summary_df)
print(f"Mean composite score: {test_results_df['Composite Score'].mean():.3f}")

In [ ]:
# Side-by-side comparison of the evaluation set and the test set
comparison_df = eval_summary_df[["Success Criterion", "Threshold"]].copy()
comparison_df["Evaluation set"] = eval_summary_df["Achieved"]
comparison_df["Test set"] = test_summary_df["Achieved"]
comparison_df["Difference (test - eval)"] = (
    comparison_df["Test set"] - comparison_df["Evaluation set"]
).round(3)

display(comparison_df)

- The test set results show how the system performs on unseen questions.
- Comparing them with the evaluation set results shows whether the system generalizes or whether the chosen settings were over-fitted to the evaluation data. A large negative difference on the test set would point to over-fitting.
- With 10 queries in each set, one query moves an average by 0.1, so small differences between the two sets are within noise.

**Note**

The next two sections, **Deployment - Backend** and **Deployment - Frontend**, set up the backend and frontend applications. The application code is the course-provided code and is kept unchanged, with two text-only exceptions made so that the deployed app matches this project:

1. In `rag_utils.py`, the prompt text now describes **hotel policies** and uses the same generation rules that were evaluated above. (The provided file carried prompt wording for a medical insurance use case.)
2. In the Streamlit app, the example question shown in the input box is a hotel question.

No logic, endpoint, dependency, or Dockerfile has been changed.

## Deployment - Backend

### Serialize the Vector Database

Before writing out the deployment files, we create a `backend_files` directory to hold all the backend components that will be pushed to GitHub and deployed in a Codespace.

In [ ]:
import os

# Create a folder to upload the backend files
os.makedirs("backend_files", exist_ok=True)

In [ ]:
# Save the index locally so it can be reloaded without rebuilding
vectorstore.save_local("backend_files/faiss_index")

print("Saved files:", os.listdir("backend_files/faiss_index"))

- The FAISS index is serialized to disk so it can be loaded directly in the backend without rebuilding from scratch.
- This saves time during deployment because the PDFs do not have to be converted and embedded again.
- The saved index includes both the vector data (`index.faiss`) and the document text and metadata (`index.pkl`).

### Define the Flask application

- `rag_utils.py` contains the core RAG functions (`retrieve`, `generate`, and `rag`) that the Flask API calls.
- These are the same functions defined and tested earlier in the notebook, written to a standalone Python file for deployment.

In [ ]:
%%writefile backend_files/rag_utils.py

from langchain_openai import ChatOpenAI
import os

# Retrieve the top-k relevant chunks from the vector store
def retrieve(query: str, k: int = 2,vectorstore=None) -> list[str]:
    results = vectorstore.similarity_search(query, k=k)

    # Return each chunk with its content and metadata
    return [
        f"Content:\n{r.page_content}\n\nMetadata:\n{r.metadata}"
        for r in results
    ]

# Define the generation function that prompts the LLM with retrieved context
def generate(query, retrieved_chunks,model) -> str:

    prompt = f"""
    You are an AI assistant answering questions about hotel policies.

    User Query:
    {query}

    Retrieved Policy Information:
    {retrieved_chunks}

    Rules:
    1. Answer only from the Retrieved Policy Information. Do not use outside knowledge or assumptions.
    2. Answer the question that was asked in the first sentence.
    3. Include every condition that changes the answer: rate plan, property, deadlines, fees, and exceptions.
    4. If the answer differs by property or rate plan and the question does not specify one, state each case.
    5. Give amounts, times, and deadlines exactly as written in the policy. Do not round or estimate.
    6. Leave out retrieved information that does not bear on the question.
    7. Keep the answer concise, clear, and easy for hotel staff to relay to a guest.
    8. If the retrieved information does not contain the answer, reply exactly:
       "The available policy documents do not contain enough information to answer this question."
    9. If retrieved passages conflict with each other, state the conflict instead of choosing one.

    Answer:
    """

    return model.invoke(prompt)

# Define the full RAG pipeline combining retrieval and generation
def rag(
    query: str,
    k: int = 2,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = 512,
    vectorstore=None
):
    # Create the generator model
    model = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        openai_api_key=os.getenv('OPENAI_API_KEY'),
        openai_api_base=os.getenv('OPENAI_API_BASE')
    )

    # Retrieve relevant chunks
    retrieved_chunks = retrieve(query=query, k=k,vectorstore=vectorstore)

    # Generate answer using retrieved chunks
    answer = generate(
        query=query,
        retrieved_chunks=retrieved_chunks,
        model=model
    )

    return answer.content, retrieved_chunks

- This file is written to `backend_files/rag_utils.py` using the `%%writefile` magic command.
- The key difference from the notebook version is that `rag` reads the API credentials from environment variables (`os.getenv`) instead of a config file, since the deployed environment uses GitHub Codespace secrets.
- The Flask application defines the REST API endpoints that the frontend calls. It loads the embedding model and FAISS index at startup, then exposes three endpoints for different levels of response detail.

In [ ]:
%%writefile backend_files/app.py

# Import necessary libraries
from flask import Flask, request, jsonify
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Import RAG utility functions
from rag_utils import retrieve, rag


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

EMBED_MODEL = "HarishMaths/Hotel-Policy-Embedding"
VECTORSTORE_PATH = "faiss_index"


# ---------------------------------------------------------
# Initialize the Flask application
# ---------------------------------------------------------

rag_api = Flask("Aurora Stay API")


# ---------------------------------------------------------
# Load the embedding model
# ---------------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


# ---------------------------------------------------------
# Load the saved FAISS vector store
# ---------------------------------------------------------

vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Vectors stored: {vectorstore.index.ntotal}")


# ---------------------------------------------------------
# Default route
# ---------------------------------------------------------

@rag_api.get("/")
def home():
    """
    Handles GET requests to the root URL.
    """
    return "Welcome to the Aurora Stay RAG API!"


# ---------------------------------------------------------
# Relevant chunks endpoint
# ---------------------------------------------------------

@rag_api.post("/v1/relevant_chunks")
def relevant_chunks():
    """
    Handles POST requests to /v1/relevant_chunks.

    Example request:
    {
        "query": "What is the cancellation policy?",
        "k": 3
    }

    Returns the top-k relevant chunks retrieved from
    the vector store.
    """

    # Get JSON data from request body
    data = request.get_json()

    # Extract parameters
    query = data["query"]
    k = data.get("k", 2)

    # Retrieve relevant chunks
    chunks = retrieve(
        query=query,
        k=k,
        vectorstore=vectorstore
    )

    # Return the retrieved chunks
    return jsonify({
        "query": query,
        "k": k,
        "relevant_chunks": chunks
    })


# ---------------------------------------------------------
# Answer with relevant chunks endpoint
# ---------------------------------------------------------

@rag_api.post("/v1/answer_with_relevant_chunks")
def answer_with_relevant_chunks():
    """
    Handles POST requests to /v1/answer_with_relevant_chunks.

    Example request:
    {
        "query": "What is the cancellation policy?",
        "k": 3,
        "model_name": "gpt-4o-mini",
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 512
    }

    Returns both the generated answer and the relevant
    chunks used to generate the answer.
    """

    # Get JSON data from request body
    data = request.get_json()

    # Extract parameters
    query = data["query"]
    k = data.get("k", 2)
    model_name = data.get("model_name", "gpt-4o-mini")
    temperature = data.get("temperature", 0.0)
    top_p = data.get("top_p", 1.0)
    max_tokens = data.get("max_tokens", 512)

    # Run the complete RAG pipeline
    answer, chunks = rag(
        query=query,
        k=k,
        model_name=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        vectorstore=vectorstore
    )

    # Return answer, chunks, and parameters used
    return jsonify({
        "query": query,
        "k": k,
        "model_name": model_name,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "relevant_chunks": chunks,
        "answer": answer
    })


# ---------------------------------------------------------
# Answer endpoint
# ---------------------------------------------------------

@rag_api.post("/v1/answer")
def answer():
    """
    Handles POST requests to /v1/answer.

    Example request:
    {
        "query": "What is the cancellation policy?",
        "k": 3,
        "model_name": "gpt-4o-mini",
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 512
    }

    Returns only the generated answer.
    """

    # Get JSON data from request body
    data = request.get_json()

    # Extract parameters
    query = data["query"]
    k = data.get("k", 2)
    model_name = data.get("model_name", "gpt-4o-mini")
    temperature = data.get("temperature", 0.0)
    top_p = data.get("top_p", 1.0)
    max_tokens = data.get("max_tokens", 512)

    # Run the complete RAG pipeline
    generated_answer, _ = rag(
        query=query,
        k=k,
        model_name=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        vectorstore=vectorstore
    )

    # Return only the answer
    return jsonify({
        "answer": generated_answer
    })


# ---------------------------------------------------------
# Run the Flask application
# ---------------------------------------------------------

if __name__ == "__main__":
    rag_api.run(debug=True)

The Flask app (`app.py`) is written to the `backend_files` folder. It defines three POST endpoints:

- `/v1/relevant_chunks` returns only the retrieved chunks for a given query.
- `/v1/answer_with_relevant_chunks` returns both the generated answer and the supporting chunks.
- `/v1/answer` returns only the generated answer.

The embedding model and FAISS index are loaded once at startup to avoid reloading them on every request. The `allow_dangerous_deserialization=True` flag is needed because FAISS indexes use pickle under the hood; this is acceptable here because the index file is one we created ourselves.

### Define the dependencies

The backend `requirements.txt` lists the Python packages needed to run the Flask API inside the Docker container.

In [ ]:
%%writefile backend_files/requirements.txt

flask==3.1.3
Werkzeug==3.1.8
requests==2.34.2
gunicorn==26.2.0
uvicorn[standard]
langchain-huggingface==1.2.2
langchain-community==0.4.2
langchain-openai==1.6.1
sentence-transformers==6.1.0
faiss-cpu==1.15.0

### Define the Dockerfile

- The Dockerfile defines how to build the backend container image.
- It installs all dependencies and pre-downloads the embedding model during the build step.

In [ ]:
%%writefile backend_files/Dockerfile

FROM python:3.13.15-slim

WORKDIR /app
COPY . .

RUN pip install --no-cache-dir --upgrade pip

RUN pip install --no-cache-dir \
    torch \
    --index-url https://download.pytorch.org/whl/cpu

RUN pip install --no-cache-dir -r requirements.txt

ENV HF_HOME=/app/huggingface

# Download the model at build time
RUN python -c "\
from langchain_huggingface import HuggingFaceEmbeddings; \
HuggingFaceEmbeddings( \
    model_name='HarishMaths/Hotel-Policy-Embedding', \
    model_kwargs={'device': 'cpu'}, \
    encode_kwargs={'normalize_embeddings': True} \
)"

# Force offline mode at runtime so it uses the cached model
ENV TRANSFORMERS_OFFLINE=1
ENV HF_HUB_OFFLINE=1

CMD ["gunicorn", "-w", "1", "--timeout", "120", "-b", "0.0.0.0:7860", "app:rag_api"]

A few important things about this Dockerfile:

- The **CPU-only build of PyTorch** is installed from a dedicated index URL to avoid pulling in the much larger GPU version.
- The **embedding model is downloaded at build time** with a `RUN` command, so the model is baked into the image.
- `TRANSFORMERS_OFFLINE=1` and `HF_HUB_OFFLINE=1` are set so the container never tries to reach Hugging Face at runtime, making startup faster and more reliable.
- **Gunicorn** runs the Flask app with one worker and a 120-second timeout on port 7860.

## Deployment - Frontend

### Define the Streamlit application

We create a separate `frontend_files` directory for the Streamlit UI components.

In [ ]:
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend_files", exist_ok=True)

- The Streamlit app provides a simple web interface where hotel staff can type their questions and get answers from the RAG system.
- It communicates with the Flask backend through HTTP requests.

In [ ]:
%%writefile frontend_files/app.py

import streamlit as st
import requests


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

BACKEND_URL = "http://localhost:7860"


# ---------------------------------------------------------
# Page configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title="Aurora Stays",
    page_icon="🏨",
    layout="wide"
)


# ---------------------------------------------------------
# Sidebar  -  Model parameters
# ---------------------------------------------------------

with st.sidebar:

    st.header("⚙️ Parameters")

    k = st.slider(
        "k (chunks to retrieve)",
        min_value=1,
        max_value=10,
        value=2,
        help="Number of relevant chunks fetched from the vector store."
    )

    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=1.0,
        value=0.0,
        step=0.1,
        help="Higher values make the output more creative; lower values make it more focused."
    )

    max_tokens = st.slider(
        "Max tokens",
        min_value=64,
        max_value=2048,
        value=512,
        step=64,
        help="Maximum number of tokens in the generated answer."
    )


# ---------------------------------------------------------
# Main area
# ---------------------------------------------------------

st.title("🏨 Aurora Stays")
st.write("Ask a question about Aurora Stays policies.")


# ---------------------------------------------------------
# Response type selection (two options only)
# ---------------------------------------------------------

response_type = st.radio(
    "Response mode",
    ["Answer only", "Answer + Relevant Chunks"],
    horizontal=True
)


# ---------------------------------------------------------
# Query input
# ---------------------------------------------------------

query = st.text_input(
    "Your question",
    placeholder="e.g., What is the cancellation deadline for a Flexible rate booking?"
)


# ---------------------------------------------------------
# Submit
# ---------------------------------------------------------

if st.button("Submit", type="primary"):

    if not query.strip():
        st.warning("Please enter a question.")

    else:

        # Pick endpoint
        if response_type == "Answer only":
            endpoint = "/v1/answer"
        else:
            endpoint = "/v1/answer_with_relevant_chunks"

        # Build request payload with parameters
        payload = {
            "query": query,
            "k": k,
            "temperature": temperature,
            "max_tokens": max_tokens
        }

        try:
            with st.spinner("Thinking…"):
                response = requests.post(
                    f"{BACKEND_URL}{endpoint}",
                    json=payload
                )

            if response.status_code == 200:

                result = response.json()

                # -- Answer only --------------------------------
                if response_type == "Answer only":

                    st.subheader("Answer")
                    st.write(result["answer"])

                # -- Answer + Relevant Chunks -------------------
                else:

                    st.subheader("Answer")
                    st.write(result["answer"])

                    st.subheader("Relevant Chunks")
                    for i, chunk in enumerate(result["relevant_chunks"], start=1):
                        with st.expander(f"Chunk {i}"):
                            st.write(chunk)

            else:
                st.error(
                    f"Unable to get a response from the RAG API. "
                    f"Status code: {response.status_code}"
                )

        except requests.exceptions.RequestException:
            st.error(
                "Unable to connect to the RAG API. "
                "Please make sure the backend service is running."
            )

The frontend app includes:

- A sidebar with adjustable parameters for `k` (number of chunks), temperature, and max tokens.
- A radio button to choose between "Answer only" and "Answer + Relevant Chunks" modes.
- A text input field for the query and a submit button.
- Error handling for both HTTP errors and connection failures.

When the user submits a question, the app sends a POST request to the appropriate backend endpoint and displays the response. In the "Answer + Relevant Chunks" mode, each supporting chunk is shown inside an expandable section, which meets the requirement to display the supporting source information with the answer.

**Usage note:** the `k` slider starts at 2. The evaluation above used the value of `K` set in this notebook, so move the slider to that value when testing the app to reproduce the evaluated behavior.

### Define the dependencies

The frontend only needs two packages: `requests` for making HTTP calls to the backend, and `streamlit` for the UI framework.

In [ ]:
%%writefile frontend_files/requirements.txt

requests==2.34.2
streamlit==1.64.0

### Define the Dockerfile

The frontend Dockerfile is simpler than the backend one since it only needs to install Streamlit and run the app.

In [ ]:
%%writefile frontend_files/Dockerfile

# Use a minimal base image with Python 3.13.15-slim installed
FROM python:3.13.15-slim

# Set the working directory inside the container
WORKDIR /app

# Copy all files from the current directory to the container's /app directory
COPY . .

# Upgrade pip
RUN pip install --no-cache-dir --upgrade pip

# Install Python dependencies listed in requirements.txt
RUN pip install -r requirements.txt

# Run the Streamlit application on port 8501
# 0.0.0.0 makes the application accessible outside the container
# XSRF protection is disabled for easier external access
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

- The frontend container runs Streamlit on port 8501 and binds to `0.0.0.0` so it is accessible outside the container.
- XSRF protection is disabled to allow external access from the Codespace forwarded URL.
- The image is lightweight since it only needs Python, Streamlit, and the requests library.

### Pushing the files to GitHub

**Before executing the cells below**

- Create a GitHub account and generate a Personal Access Token by following the course document *Introduction to GitHub and GitHub Codespaces and Creating GitHub Account and Generating Personal Access Token*.
- **In Google Colab:** open the Secrets pane (the key icon in the left sidebar), click **+ Add new secret**, name it `git_token`, paste the token as the value, and turn on **Notebook access**.
- **In VS Code / Jupyter:** either set an environment variable named `GITHUB_TOKEN` before starting the kernel, or simply run the cell and paste the token into the prompt that appears.

The token is never written into the notebook, which keeps it out of the saved file and out of version control.

In [ ]:
# Your GitHub Personal Access Token
if IN_COLAB:
    # Read the token from Colab's secure Secrets storage.
    # You may be prompted to grant access. Click "Grant access" to continue.
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('git_token')   # Replace 'git_token' if your secret has a different name
else:
    # Local kernel: use the GITHUB_TOKEN environment variable, or ask for the token
    import getpass
    GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") or getpass.getpass("GitHub Personal Access Token: ")

if not GITHUB_TOKEN:
    raise ValueError("No GitHub token found. Add it as described above and re-run this cell.")
print("GitHub token loaded.")

The next cell:

1. **Authenticates with GitHub** using the token.
2. **Creates the repository** `aurora-stay-qna` if it does not already exist.
3. **Pushes the deployment files**: backend files (`app.py`, `rag_utils.py`, `requirements.txt`, `Dockerfile`, and the FAISS index) go into `backend/`, and frontend files go into `frontend/`.

The cell can be re-run safely: existing files are updated instead of duplicated.

In [ ]:
# Name for the repository
REPO_NAME = "aurora-stay-qna"

# Authenticate with GitHub
from github import Github
import os

g = Github(GITHUB_TOKEN)
user = g.get_user()

print(f"Authenticated as: {user.login}")


# ---------------------------------------------------------
# Get existing repository or create a new one
# ---------------------------------------------------------

try:
    # Check if repository already exists
    repo = user.get_repo(REPO_NAME)
    print(f"Repository already exists: {repo.html_url}")

except Exception:
    # Repository does not exist, so create it
    repo = user.create_repo(
        REPO_NAME,
        description="Aurora RAG",
        private=False,
        auto_init=True
    )

    print(f"Repository created successfully: {repo.html_url}")


# ---------------------------------------------------------
# Files to push
# ---------------------------------------------------------

files_to_push = {
    "backend/app.py": "backend_files/app.py",
    "backend/rag_utils.py": "backend_files/rag_utils.py",
    "backend/requirements.txt": "backend_files/requirements.txt",
    "backend/Dockerfile": "backend_files/Dockerfile",

    "frontend/app.py": "frontend_files/app.py",
    "frontend/requirements.txt": "frontend_files/requirements.txt",
    "frontend/Dockerfile": "frontend_files/Dockerfile",
}


# ---------------------------------------------------------
# Function to upload or update a file
# ---------------------------------------------------------

def upload_or_update_file(repo, repo_path, local_path):

    with open(local_path, "rb") as f:
        content = f.read()

    try:
        # Check if the file already exists
        existing_file = repo.get_contents(
            repo_path,
            ref="main"
        )

        # File exists -> update it
        repo.update_file(
            path=repo_path,
            message=f"Update {repo_path}",
            content=content,
            sha=existing_file.sha,
            branch="main"
        )

        print(f"Updated: {repo_path}")

    except Exception:
        # File does not exist -> create it
        repo.create_file(
            path=repo_path,
            message=f"Add {repo_path}",
            content=content,
            branch="main"
        )

        print(f"Uploaded: {repo_path}")


# ---------------------------------------------------------
# Upload / update individual files
# ---------------------------------------------------------

for repo_path, local_path in files_to_push.items():

    upload_or_update_file(
        repo,
        repo_path,
        local_path
    )

# ---------------------------------------------------------
# Upload / update FAISS index folder
# ---------------------------------------------------------

faiss_folder = "backend_files/faiss_index"

if os.path.exists(faiss_folder):

    for root, dirs, files in os.walk(faiss_folder):

        for file_name in files:

            # Full local path
            local_path = os.path.join(
                root,
                file_name
            )

            # Path relative to backend_files/
            relative_path = os.path.relpath(
                local_path,
                "backend_files"
            )

            # GitHub repository path
            repo_path = os.path.join(
                "backend",
                relative_path
            ).replace("\\", "/")

            upload_or_update_file(
                repo,
                repo_path,
                local_path
            )

else:
    print(
        f"Warning: FAISS folder not found: {faiss_folder}"
    )


print("\nAll files pushed successfully!")
print(f"Repository: {repo.html_url}")

Once all the files have been pushed, the next step is to add the secrets (`OPENAI_API_KEY` and `OPENAI_API_BASE`) to GitHub Codespaces so that the backend can connect to OpenAI. For this, refer to the course document *Adding Secrets to GitHub Codespaces*.

**Note:** The link to your GitHub repository is displayed as the output of the previous cell.

Then refer to the course document *Deploying Frontend and Backend on GitHub Codespaces* for the steps to deploy the Streamlit application and see it in action.

## Deployment - Inference

Before executing the code cells below, ensure the following:

- GitHub Codespaces is up and running.
- Both the frontend and backend applications are running in GitHub Codespaces, and the ports are set to **Public**.
- The forwarded URL of the backend (port 7860) is copied from the Codespace's **Ports** tab.

In [ ]:
# Set the base URL for the deployed Flask API running in GitHub Codespaces.
# Replace the value below with your Codespace forwarded URL for port 7860.
model_root_url = "https://YOUR-CODESPACE-NAME-7860.app.github.dev"

# Remove any trailing slash so the endpoint paths join correctly
model_root_url = model_root_url.rstrip("/")

if "YOUR-CODESPACE-NAME" in model_root_url:
    raise ValueError(
        "Replace model_root_url with the forwarded URL of port 7860 from your Codespace's Ports tab."
    )
print("Backend URL:", model_root_url)

The URL looks something like `https://organic-space-abcd1234-7860.app.github.dev`.

This URL is a tunnel that Codespaces creates to route external HTTP requests into the Codespace and on to the Flask container running on port 7860.

In [ ]:
# Confirm the backend is reachable before sending queries
health = requests.get(model_root_url + "/", timeout=60)
print(health.status_code, health.text)

In [ ]:
model_chunks_url = model_root_url + "/v1/relevant_chunks"  # Endpoint for getting relevant chunks

Predictions are served through the Flask endpoints created earlier, so we call those endpoints:

`@rag_api.post('/v1/relevant_chunks')`

In [ ]:
model_ans_chunks_url = model_root_url + "/v1/answer_with_relevant_chunks"  # Endpoint for getting the answer with relevant chunks

`@rag_api.post('/v1/answer_with_relevant_chunks')`

This is online (real-time) inference: a single request is sent to the API and the response comes back immediately.

- The query and parameters are sent as a JSON payload in a POST request.
- The backend processes the request and returns the result as a JSON payload.

### Deployment - Inference (Relevant chunks)

In [ ]:
# Define the test payload with a sample hotel policy question and the number of chunks to retrieve
payload = {
    "query": "What is the charge for smoking in a non-smoking room?",  # A hotel policy question
    "k": 4,                                                            # Number of chunks to retrieve
}

In [ ]:
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_chunks_url, json=payload)

In [ ]:
# Check the HTTP status of the response
response

- `<Response [200]>` is an HTTP status code.
- It indicates that the request was successful and the server processed it without any problems.

In [ ]:
# Print the retrieved chunks returned by the backend
chunks_result = response.json()

print("Query:", chunks_result["query"])
print("k:", chunks_result["k"])
for i, chunk in enumerate(chunks_result["relevant_chunks"], start=1):
    print(f"\n--- Relevant chunk {i} ---")
    print(chunk)

- The response contains the original query, the value of `k`, and the top-k retrieved chunks.
- Each chunk includes the text content and metadata showing which source document it came from.
- This confirms the retrieval endpoint is working on the deployed backend.

### Deployment - Inference (Answer with relevant chunks)

The parameters match the configuration that was evaluated: the same model, `k`, temperature, top-p, and token limit.

In [ ]:
# Define the test payload with the query and model parameters for the answer endpoint
payload = {
    "query": "A guest booked an Advance Purchase rate and wants to cancel. Can they get a refund?",  # A hotel policy question
    "k": 4,                       # Number of chunks to retrieve
    "model_name": "gpt-4o-mini",  # LLM used to generate the answer
    "temperature": 0,             # Low value for consistent responses
    "top_p": 1,                   # Value close to 1 for balanced token selection
    "max_tokens": 512             # Limit based on the expected answer length
}

**Note:** The cell below may take some time to execute, typically between 1 and 3 minutes.

In [ ]:
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_ans_chunks_url, json=payload)

In [ ]:
# Check the HTTP status of the response
response

- `<Response [200]>` is an HTTP status code.
- It indicates that the request was successful and the server processed it without any problems.

In [ ]:
# Print the generated answer and the supporting chunks
answer_result = response.json()

print("Query:", answer_result["query"])
print("\nAnswer:\n", answer_result["answer"])

for i, chunk in enumerate(answer_result["relevant_chunks"], start=1):
    print(f"\n--- Relevant chunk {i} ---")
    print(chunk)

- The response includes both the generated answer and the relevant chunks used to produce it.
- This is the same payload the Streamlit frontend receives when the user selects "Answer + Relevant Chunks" mode.
- Having the answer and its supporting evidence in a single response lets hotel staff verify the information before sharing it with guests.

**IMPORTANT:** Once the previous cell has been executed and the application is no longer required, refer to the course document *Deleting Codespace* to delete the Codespace, as it may continue to consume resources.

## Business Insights and Recommendations

### Results against the success criteria

The cell below turns the evaluation and test results from this run into a plain-language summary, so the figures in this section always match the tables above.

In [ ]:
# Build a plain-language summary of the results from this run
WEEKLY_QUERIES = 1200   # Guest queries handled per week (from the business context)


def describe_results(result_df: pd.DataFrame, label: str) -> list[str]:
    lines = [f"**{label} ({len(result_df)} queries)**", ""]

    # Each metric against its threshold
    gaps = {}
    for column, threshold in THRESHOLDS.items():
        achieved = result_df[column].mean()
        gaps[column] = achieved - threshold
        status = "meets" if achieved >= threshold else "is below"
        lines.append(
            f"- {column.replace(' Score', '')}: **{achieved:.2f}**, which {status} "
            f"the {threshold:.2f} threshold."
        )

    # Share of queries that reach the Auto Answer cutoff
    auto_count = int((result_df["Composite Score"] >= AUTO_ANSWER_CUTOFF).sum())
    auto_share = auto_count / len(result_df)
    status = "meets" if auto_share >= AUTO_ANSWER_TARGET else "is below"
    lines.append(
        f"- Queries confidently answered: **{auto_count} of {len(result_df)} ({auto_share:.0%})**, "
        f"which {status} the {AUTO_ANSWER_TARGET:.0%} target."
    )

    # The metric with the least headroom (or the largest shortfall)
    weakest = min(gaps, key=gaps.get)
    lines.append(
        f"- Metric with the least headroom: **{weakest.replace(' Score', '')}** "
        f"({gaps[weakest]:+.2f} against its threshold)."
    )

    # The query the system handled worst
    worst = result_df.loc[result_df["Composite Score"].idxmin()]
    lines.append(
        f"- Lowest-scoring query (composite {worst['Composite Score']:.2f}): \"{worst['Question']}\""
    )
    lines.append("")
    return lines


summary_lines = describe_results(eval_results_df, "Evaluation set")
summary_lines += describe_results(test_results_df, "Test set")

# Illustration of what the test-set share would mean at weekly volume
test_auto_share = (test_results_df["Composite Score"] >= AUTO_ANSWER_CUTOFF).mean()
summary_lines.append(
    f"**Illustration at weekly volume:** if the test-set share held across all "
    f"{WEEKLY_QUERIES:,} weekly queries, about **{round(test_auto_share * WEEKLY_QUERIES):,}** "
    f"queries per week would not need a manual lookup. This is an extrapolation from "
    f"{len(test_results_df)} test queries and is indicative only."
)

display(Markdown("\n".join(summary_lines)))

In [ ]:
# Queries that fell below the Auto Answer cutoff, across both sets, for error analysis
below_cutoff = pd.concat(
    [eval_results_df.assign(Set="Evaluation"), test_results_df.assign(Set="Test")],
    ignore_index=True,
)
below_cutoff = below_cutoff[below_cutoff["Composite Score"] < AUTO_ANSWER_CUTOFF]

if below_cutoff.empty:
    print("Every query in both sets reached the Auto Answer cutoff.")
else:
    display(below_cutoff.sort_values("Composite Score"))

### Insights

- **What the POC shows.** The summary above states, for this run, which success criteria are met and what share of queries the system can answer without a manual lookup. That share is the answer to the POC's main question: whether the system can reliably handle a meaningful proportion of queries.
- **Retrieval sets the ceiling.** Faithfulness and Answer Relevancy depend on the generation rules, which are easy to change. Contextual Recall and Contextual Precision depend on chunking and on how many chunks are retrieved, and they pull against each other: a larger `k` finds more of the answer for multi-document questions but brings in more unrelated text. The queries listed below the cutoff show which side is costing more in this run.
- **Multi-document questions are the hard cases.** Topics such as pets, service animals, and cancellation-plus-refund are split across documents. These are the 40% of queries that take staff 8 to 12 minutes today, so they are where the system has the most to gain and the most room to fall short.
- **Declining is the safe failure.** The generation rules tell the system to say so when the documents do not cover a question, and the out-of-scope check in the generation section tests this. For a hotel, a declined answer that goes to manual lookup costs far less than a confident wrong answer about a fee or a refund.
- **The evidence base is small.** Each set has 10 queries, so one query changes an average by 0.1, and judge scores vary between runs. The results indicate direction; they are not yet a precise estimate of production performance.

### Recommendations

1. **Run an internal pilot with sources always shown.** Deploy to front-desk and reservations staff at a few properties with the "Answer + Relevant Chunks" view as the default, so staff can verify each answer against the policy text before relaying it to a guest.
2. **Grow the evaluation set before any wider rollout.** Collect 150 to 200 real staff queries with verified answers, balanced across single-document and multi-document questions and across properties. Repeat each evaluation several times and report the average, so that go/no-go decisions do not rest on 10 queries and one judge run.
3. **Design live routing around what can be measured live.** The composite score uses Contextual Recall, which needs a known correct answer and is therefore only available offline. For production routing, calibrate a confidence signal that needs no reference answer (faithfulness, answer relevancy, and retrieval similarity) against the labeled set, and use it to decide between Auto Answer and Manual Lookup.
4. **Always send sensitive topics to a person.** Questions involving safety threats, discrimination, privacy, charge disputes, or legal and media requests fall under the escalation matrix in the conduct guidelines. Route these to the named escalation path regardless of score.
5. **Prepare the knowledge base for growth.** With 3 to 4 new properties each year, tag every chunk with its property and policy area so retrieval can filter by property, keep each table together with its header row when chunking, and re-test whenever a property is added.
6. **Give the documents an owner and a refresh process.** Assign an owner per policy document, record an effective date, and rebuild the index whenever a policy changes. A stale index would produce faithful answers to an outdated policy, which the metrics here cannot detect.
7. **Secure the application before wider use.** In this POC the API has no authentication, the Codespace ports are public, and the repository is public. Before a pilot, add staff sign-in, make the repository private, and host the service inside the company network.
8. **Track the business outcome.** During the pilot, measure the share of queries answered without manual lookup, how often staff override the answer, and the error rate against the current 15% to 20% baseline. Review the overridden and below-cutoff queries each month to find missing or unclear policy content.
9. **Use it for onboarding.** With 45% annual staff turnover, a tool that shows the policy source next to every answer also helps new staff learn where each policy lives, which reduces reliance on knowledge held by experienced colleagues.

### Conclusion

This POC delivers a working end-to-end system: policy PDFs are converted and indexed, staff questions are answered from retrieved policy text with the sources shown, quality is measured against explicit thresholds, and the application is deployed behind an API with a simple UI. The results summary above shows how this run compares with the success criteria. The recommended next step is a controlled internal pilot with a larger labeled query set, property-aware retrieval, and access controls, with the share of queries answered without manual lookup as the headline measure.